`Transform` pipelines can be used with any data loader and any dataset. They are simply functions that take as input an `AtomArray` (which is often the output of `AtomWorks.io`) and output `PyTorch` tensors ready for ingestion by a model.

However, most users will not want to build datasets from scratch. For convenience, we provide pre-built datasets and dataloaders that play well with `Transform` pipelines as well, roughly adhering to [Torchvision](https://docs.pytorch.org/vision/stable/datasets.html) conventions.

We demonstrate below a couple of different ways to connect a `Transform` pipeline with arbitray datasets and connect them with trivial `Transform` pipelines.

# Datasets in AtomWorks

### Using a Folder of CIF/PDB Files as a Dataset

The simplest way to use AtomWorks with a Dataset is to create a `Dataset` and `Sampler` pointed to a dictory of structural files (e.g., PDB, CIF).

NOTE: All AtomWorks Datasets require a `name` attribute to support many of the logging/debugging features that are supplied out-of-the-box.

In [61]:
from atomworks.ml.datasets.datasets import FileDataset

# To setup the test pack, if not already, run `atomworks setup tests`
# You may need to then adjust your `.env` file to point to the test data (e.g., PDB_MIRROR_PATH=tests/data/pdb)
dataset = FileDataset.from_directory(directory="../../tests/data/pdb", name="example_pdb_dataset")

Let's explore the dataset a tiny bit.

In [62]:
# Count the number of examples in the dataset
print(f"Dataset has {len(dataset)} examples.")

# Print the raw data of the first 5 examples
for i, example in enumerate(dataset):
    if i >= 5:
        break
    print(f"Example {i + 1}: {example}")

Dataset has 176 examples.
Example 1: ../../tests/data/pdb/01/101m.cif.gz
Example 2: ../../tests/data/pdb/04/104d.cif.gz
Example 3: ../../tests/data/pdb/12/112m.cif.gz
Example 4: ../../tests/data/pdb/33/133d.cif.gz
Example 5: ../../tests/data/pdb/55/155c.cif.gz


At a high level, to train models with AtomWorks, we need typically need a Dataset that:

(1) Takes as input an item index and returns the corresponding example information; typically includes:
a. Path to a structural file saved on disk (`/path/to/dataset/my_dataset_0.cif`)
b. Additional item-specific metadata (e.g., class labels)

(2) Pre-loads structural information from the returned example into an `AtomArray` and assembles inputs for the Transform pipeline

(3) Feed the input dictionary through a Transform pipeline and returns the result


So far, the `FileDataset` we initialized only accomplishes (1) from above - returning the raw data.

To accomplish (2), we can additionally pass a loading function at dataset initialization that takes the raw example data as input and returns a pre-processed ready for a Transform pipeline.

In most cases, this will involve using `parse` or `load_any` from `AtomWorks.io` to build an `AtomArray`, which is the common language of our `Transform` library.

In [63]:
from atomworks.io import parse

def simple_loading_fn(raw_data) -> dict:
    parse_output = parse(raw_data)
    return {"atom_array": parse_output["assemblies"]["1"][0]}

dataset_with_loading_fn = FileDataset.from_directory(
    directory="../../tests/data/pdb",
    name="example_pdb_dataset",
    loader=simple_loading_fn
)
output = dataset_with_loading_fn[1]
print(f"Output AtomArray has {len(output['atom_array'])} atoms!")

Output AtomArray has 798 atoms!


Next up is adding in a pipeline. Let's create a simple one with a dramatic crop.

In [64]:
from atomworks.ml.transforms.base import Compose
from atomworks.ml.transforms.crop import (
    CropSpatialLikeAF3,
)
from atomworks.ml.transforms.atom_array import (
    AddGlobalAtomIdAnnotation,
)
from atomworks.ml.transforms.atomize import AtomizeByCCDName
from atomworks.constants import STANDARD_AA

pipe = Compose(
    [
        # (We need to add these transforms before we can crop)
        AddGlobalAtomIdAnnotation(),
        AtomizeByCCDName(atomize_by_default=True, res_names_to_ignore=STANDARD_AA),
        # Crop to 20 tokens (which in this case is number ambino acids/nucleic acid bases + number of small molecule atoms)
        CropSpatialLikeAF3(crop_size=20)
    ],
    track_rng_state=False,
)

Just like with the loading function, we can also pass a composed `Transform` pipeline to our datasets.

In [65]:
dataset_with_loading_fn_and_transforms = FileDataset.from_directory(
    directory="../../tests/data/pdb",
    name="example_pdb_dataset",
    loader=simple_loading_fn,
    transform=pipe
)

In [66]:
from atomworks.io.utils.visualize import view
pipeline_output = dataset_with_loading_fn_and_transforms[0]  # This will trigger the loading function and print the row information

view(pipeline_output["atom_array"])


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

And indeed, we have a cropped example!

We will then sample uniformly (with or without replacement) from this dataset during training. Such a simple application may be appropriate for many fine-tuning cases such as distillation.

The only "gotcha" outside of normal PyTorch sampling is that you'll need to implement a default collate function (which could simply be the identity) so long as your output dictionary contains an `AtomArray`.


In [67]:
from torch.utils.data import RandomSampler, DataLoader

sampler = RandomSampler(dataset_with_loading_fn_and_transforms)
loader = DataLoader(
    dataset=dataset_with_loading_fn_and_transforms,
    sampler=sampler,
    collate_fn=lambda x: x  # Identity collate: returns the batch as-is
)

for i, example in enumerate(loader):
    # (Since we now have a batch dimension, we need the extra indexing dimension)
    print(f"Example: {i}, Length of AtomArray: {len(example[0]["atom_array"])}")
    if i > 2:
        break


Example: 0, Length of AtomArray: 282
Example: 1, Length of AtomArray: 20
Example: 1, Length of AtomArray: 20
Example: 2, Length of AtomArray: 324
Example: 3, Length of AtomArray: 20
Example: 2, Length of AtomArray: 324
Example: 3, Length of AtomArray: 20


For more complicated sampling strategies, including distributed sampling for multi-GPU training, see the API documentation for `samplers.py`, and the tests in `test_samplers.py`

### Tabular Datasets

So far, we have seen how to make and use simple datasets with just paths. But in many cases we also have other pre-processed data, class labels, sampling weights, etc. etc.

We thus support instantiating datasets from tabular sources stored on disk.

We only have implemented a `PandasDataset`; however, any tabular format (e.g., `PolarsDataset`) could be similarly implemented without difficulty should the need arise (PR's welcome!)


#### PandasDataset

The `PandasDataset` class requires a couple of  arguments:
- `data`: Either a pandas DataFrame or path to a CSV/Parquet file containing the tabular data. Each row represents one example.
- `name`: Descriptive name for this dataset, just as in `FileDataset` and all AtomWorks `Dataset` classes. Used for debugging and some downstream functions when using nested datasets.

Again, we can also pass a `transform` pipeline and `loader`:
- `transform`: Transform pipeline to apply to loaded data.
- `loader`: Optional function to process raw DataFrame rows into Transform-ready format.

There's also a few other `PandasDataset`-specific arguments to note:
- `filters`: Optional list of pandas query strings to filter the data. Applied in order during initialization.
- `columns_to_load`: Optional list of column names to load when reading from a file. If None, all columns are loaded. Can dramatically reduce memory usage and load time if loading from a columnar format like Parquet.

We will start by exploring an example metatada dataframe, then load it into a `PandasDataset`.

In [83]:
from atomworks.ml.utils.io import read_parquet_with_metadata
interfaces_metadata_parquet_path = "../../tests/data/ml/pdb_interfaces/metadata.parquet"
interfaces_df = read_parquet_with_metadata(interfaces_metadata_parquet_path)
interfaces_df.head()

,pdb_id,path,assembly_id,clash_severity,resolution,deposition_date,release_date,method,num_polymer_pn_units,num_resolved_atoms_in_processed_assembly,...,"pn_unit_2_protein_cluster_(id:0,4)(cov:0,8)(cov_mode:0)_rep_seq_hash","pn_unit_1_protein_cluster_(id:1,0)(cov:0,8)(cov_mode:0)_rep_seq_hash","pn_unit_2_protein_cluster_(id:1,0)(cov:0,8)(cov_mode:0)_rep_seq_hash","pn_unit_1_nucleic_acid_cluster_(id:0,4)(cov:0,8)(cov_mode:0)_rep_seq_hash","pn_unit_2_nucleic_acid_cluster_(id:0,4)(cov:0,8)(cov_mode:0)_rep_seq_hash","pn_unit_1_nucleic_acid_cluster_(id:1,0)(cov:0,8)(cov_mode:0)_rep_seq_hash","pn_unit_2_nucleic_acid_cluster_(id:1,0)(cov:0,8)(cov_mode:0)_rep_seq_hash",pn_unit_1_cluster,pn_unit_2_cluster,cluster
0,303d,03/303d.cif.gz,1,no-clash,2.2,1996-06-26,1997-01-20,X-RAY_DIFFRACTION,2,519,...,None,None,None,2b1bb060328,2b1bb060328,2b1bb060328,2b1bb060328,2b1bb060328,2b1bb060328,2b1bb060328+2b1bb060328
1,303d,03/303d.cif.gz,1,no-clash,2.2,1996-06-26,1997-01-20,X-RAY_DIFFRACTION,2,519,...,None,None,None,2b1bb060328,None,2b1bb060328,None,2b1bb060328,RO2,2b1bb060328+RO2
2,303d,03/303d.cif.gz,1,no-clash,2.2,1996-06-26,1997-01-20,X-RAY_DIFFRACTION,2,519,...,None,None,None,2b1bb060328,None,2b1bb060328,None,2b1bb060328,MG,2b1bb060328+MG
3,303d,03/303d.cif.gz,1,no-clash,2.2,1996-06-26,1997-01-20,X-RAY_DIFFRACTION,2,519,...,None,None,None,2b1bb060328,None,2b1bb060328,None,2b1bb060328,RO2,2b1bb060328+RO2
4,104d,04/104d.cif.gz,1,no-clash,NaN,1994-12-16,1995-03-31,SOLUTION_NMR,2,494,...,None,None,None,269157d4882,269157d4882,269157d4882,269157d4882,269157d4882,269157d4882,269157d4882+269157d4882


In [80]:
from atomworks.ml.utils.io import to_parquet_with_metadata

# Replace the path cell by removing the leading /projects/ml/frozen_pdb_copies/2024_12_01_pdb/ from the paths in the metadata DataFrame
interfaces_df["path"] = interfaces_df["path"].str.replace(
    r"^/projects/ml/frozen_pdb_copies/2024_12_01_pdb/", "", regex=True
)
interfaces_df.head()

# Add the absolute path of the pwd of ../../tests/data/pdb/ to the attrs of the df as base_path
import os
absolute_base_path = os.path.abspath("../../tests/data/pdb/")
interfaces_df.attrs["base_path"] = absolute_base_path
print(interfaces_df.attrs["base_path"])

# Save the modified DataFrame back to a new parquet file with metadata
to_parquet_with_metadata(interfaces_df, interfaces_metadata_parquet_path)

/Users/nscorley/Documents/Projects/atomworks-dev/tests/data/pdb


This dataframe includes a row for every interfaces between two `pn_units` (essentially, chains) in the Protein Data Bank. For illustration purposes, however, we're loading the test dataframe, which only includes information for a small subset of the full PDB.

The complete dataframes can be downloaded with `atomworks setup metatada` and will described in greater detail elsewhere in the documentation.

For our purposes, note that we have a `path` column that points to a `.cif` file stored on disk, and an `example_id` column which is unique across every row in the dataset. 
> NOTE: Because a given PDB ID may contain many interfaces and thus may appear multiple times in our dataset, we must also incorporate the `assembly_id` and the `pn_unit_iids` of the two interacting chains within the `example_id`.

In [68]:
from atomworks.ml.datasets import PandasDataset

inferr

dataset = PandasDataset(

SyntaxError: incomplete input (71832165.py, line 3)

### Using Functional Loaders

We also provide functional alternatives to the class-based metadata parsers. These loaders are composable functions that can be used with `PandasDataset` to process DataFrame rows into Transform-ready format.

The functional approach follows the same patterns as your new dataset design - clean separation of concerns with composable building blocks.

In [72]:
# Import the functional loader factory
from atomworks.ml.datasets.loaders import loader_with_query_pn_units

# Create a sample DataFrame that represents interfaces dataset
import pandas as pd

sample_data = pd.DataFrame({
    'example_id': ['interface_1', 'interface_2'],
    'path': ['structures/1abc.cif', 'structures/2def.cif'],
    'pn_unit_1_iid': ['A_1', 'A_1'], 
    'pn_unit_2_iid': ['B_1', 'C_1'],
    'assembly_id': ['1', '1'],
    'resolution': [1.5, 2.0],
    'method': ['X-RAY_DIFFRACTION', 'X-RAY_DIFFRACTION']
})

print("Sample interfaces DataFrame:")
print(sample_data)

# Create a functional loader equivalent to GenericDFParser
loader = create_generic_df_loader(
    pn_unit_iid_colnames=['pn_unit_1_iid', 'pn_unit_2_iid'],
    assembly_id_colname='assembly_id',
    base_path='/data/pdb',
    extension='.gz'
)

# Test the loader on a single row
row = sample_data.iloc[0]
result = loader(row)

print("\nFunctional loader output:")
for key, value in result.items():
    print(f"  {key}: {value}")
    
print(f"\nNote how extra_info contains: {list(result['extra_info'].keys())}")
print("(Excluding the columns we used for specific fields)")

Sample interfaces DataFrame:
    example_id                 path pn_unit_1_iid pn_unit_2_iid assembly_id  \
0  interface_1  structures/1abc.cif           A_1           B_1           1   
1  interface_2  structures/2def.cif           A_1           C_1           1   

   resolution             method  
0         1.5  X-RAY_DIFFRACTION  
1         2.0  X-RAY_DIFFRACTION  

Functional loader output:
  example_id: interface_1
  path: /data/pdb/structures/1abc.gz
  assembly_id: 1
  query_pn_unit_iids: ['A_1', 'B_1']
  extra_info: {'resolution': 1.5, 'method': 'X-RAY_DIFFRACTION'}

Note how extra_info contains: ['resolution', 'method']
(Excluding the columns we used for specific fields)


In [74]:
# Import PandasDataset for this example
from atomworks.ml.datasets.datasets import PandasDataset

# Now use the functional loader with PandasDataset
dataset = PandasDataset(
    data=sample_data,
    name="interfaces_with_functional_loader", 
    loader=loader  # Pass our functional loader
)

print(f"Created dataset with {len(dataset)} examples")

# Test accessing an example
example = dataset[0] 
print("\nExample from PandasDataset with functional loader:")
for key, value in example.items():
    print(f"  {key}: {value}")

print("\nThis functional approach provides:")
print("✅ Composable building blocks (build_metadata_hierarchy, build_structure_path, etc.)")
print("✅ No class instantiation needed")
print("✅ Clear separation of concerns")
print("✅ Same output as GenericDFParser but with functional programming patterns")
print("✅ Easy to test individual components")

Created dataset with 2 examples

Example from PandasDataset with functional loader:
  example_id: interface_1
  path: /data/pdb/structures/1abc.gz
  assembly_id: 1
  query_pn_unit_iids: ['A_1', 'B_1']
  extra_info: {'resolution': 1.5, 'method': 'X-RAY_DIFFRACTION'}

This functional approach provides:
✅ Composable building blocks (build_metadata_hierarchy, build_structure_path, etc.)
✅ No class instantiation needed
✅ Clear separation of concerns
✅ Same output as GenericDFParser but with functional programming patterns
✅ Easy to test individual components


In [75]:
# Compare with the class-based approach for verification
from atomworks.ml.datasets.parsers.default_metadata_row_parsers import GenericDFParser

# Create equivalent class-based parser
parser = GenericDFParser(
    pn_unit_iid_colnames=['pn_unit_1_iid', 'pn_unit_2_iid'],
    assembly_id_colname='assembly_id', 
    base_path='/data/pdb',
    extension='.gz'
)

# Compare outputs
row = sample_data.iloc[0]
functional_result = loader(row)
parser_result = parser._parse(row)

print("Functional loader and GenericDFParser produce identical results:")
print(f"Results are equal: {functional_result == parser_result}")

if functional_result == parser_result:
    print("✅ Perfect equivalence achieved!")
else:
    print("❌ Results differ - this shouldn't happen")
    print("Functional:", functional_result)
    print("Parser:", parser_result)

Functional loader and GenericDFParser produce identical results:
Results are equal: True
✅ Perfect equivalence achieved!


In [ ]:
# Backward compatibility demonstration
print("=== Backward compatibility with StructuralFileDataset ===")

# Import the deprecated class
from atomworks.ml.datasets.datasets import StructuralFileDataset

# This will show a deprecation warning but still work
import warnings

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    old_dataset = StructuralFileDataset(["file1.pdb", "file2.cif"])

    if w:
        print(f"Warning: {w[0].message}")

print(f"Old API still works: {type(old_dataset[0])}")
print(f"Returns DataFrame rows: {old_dataset[0].name}")
print()

# Show that the APIs are now consistent
print("=== API Consistency Summary ===")
print("FileDataset (simple mode)     -> returns: str (file path)")
print("FileDataset (dataframe mode)  -> returns: pandas.Series")
print("PandasDataset                 -> returns: pandas.Series")
print("Both have .data property      -> returns: pandas.DataFrame")
print("Both support id_to_idx/idx_to_id methods")
print("Both support 'in' operator for ID checking")

=== Backward compatibility with StructuralFileDataset ===
Old API still works: <class 'pandas.core.series.Series'>
Returns DataFrame rows: file1

=== API Consistency Summary ===
FileDataset (simple mode)     -> returns: str (file path)
FileDataset (dataframe mode)  -> returns: pandas.Series
PandasDataset                 -> returns: pandas.Series
Both have .data property      -> returns: pandas.DataFrame
Both support id_to_idx/idx_to_id methods
Both support 'in' operator for ID checking


In many applications, we may want more nuanced dataset schemes. For example, when training on the PDB, we typically want to sample at the chain-level rather than the entry-level (since we are cropping, the two are distinct). We may also want to provide additional information other than the raw CIF file (e.g., class labels) to be used by the model during training. 

In AtomWorks, we accomplish this customization by building `parquet` files which contain a row for every possible example in the dataset. For example, in our `pn_units` (e.g., chains) dataset, we have a row for every `pn_unit` in the PDB.

We will briefly describe the `pn_unit` and `interfaces` datasets now, as these are the most important two.

In [ ]:
# You can download the pre-build metadata parquets using `atomworks setup metadata /path/to/your/metadata/directory/`
METADATA_DIR = "/path/to/your/metadata/directory/"
METADATA_DIR = "/Users/nscorley/Documents/Projects"